In [ ]:
# import os

# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import jax
import jax.numpy as jnp

jax.config.update("jax_compilation_cache_dir", "./jax-caches")
jax.config.update("jax_persistent_cache_min_entry_size_bytes", -1)
jax.config.update("jax_persistent_cache_min_compile_time_secs", 0)

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath("."))
sys.path.insert(0, os.path.abspath("."))
sys.path.append(os.path.abspath("../"))
sys.path.append(os.path.abspath("../../"))

from desc import set_device
set_device("gpu")

In [ ]:
import numpy as np
np.set_printoptions(linewidth=np.inf, precision=4, suppress=True, threshold=sys.maxsize)
import matplotlib.pyplot as plt
%matplotlib inline
import plotly.graph_objects as go

In [ ]:
import desc

from desc.basis import *
from desc.backend import *
from desc.compute import *
from desc.coils import *
from desc.equilibrium import *
from desc.examples import *
from desc.grid import *
from desc.geometry import *
from desc.io import *

from desc.objectives import *
from desc.objectives.objective_funs import *
from desc.objectives.getters import *
from desc.objectives.normalization import compute_scaling_factors
from desc.objectives.utils import *
from desc.optimize._constraint_wrappers import *

from desc.transform import Transform
from desc.plotting import *
from desc.optimize import *
from desc.perturbations import *
from desc.profiles import *
from desc.compat import *
from desc.utils import *
from desc.magnetic_fields import *
from desc.particles import *
from diffrax import *

from desc.__main__ import main
from desc.vmec_utils import vmec_boundary_subspace
from desc.input_reader import InputReader
from desc.continuation import solve_continuation_automatic
from desc.compute.data_index import register_compute_fun
from desc.optimize.utils import solve_triangular_regularized

print_backend_info()

In [ ]:
from extra_objectives import CoilBounds, SurfaceMatch

In [ ]:
# extra plotting functions
from plotting_yge import plot_grid_3d, plot_coil_and_surfaces

# About this Notebook

We will optimize an umbilic coil for HBT hybrid design.

In [ ]:
# eq0 = load("eq_final2.h5")
eq0 = load("./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb.h5")
field0 = load("./coils/coils-eq-HBT_105995_06-rev-curr-stage2-fb.h5")
all_HBT_coils_wo_tf = field0[1:]
all_HBT_coils_wo_tf = MixedCoilSet(all_HBT_coils_wo_tf)
tf = ToroidalMagneticField(B0=0.32, R0=0.92)
field0 = [tf, all_HBT_coils_wo_tf]
use_coil2 = True

## First get N!=0 and NFP=2 Solution

In [ ]:
field = field0.copy()

In [ ]:
eq = eq0.copy()
eq.change_resolution(L=16, M=16, L_grid=24, M_grid=24, N=3, N_grid=6, NFP=2, sym=True)
# eq.change_resolution(N=2, N_grid=4, NFP=1, sym=True)
coil_grid = LinearGrid(N=50)
eval_grid = LinearGrid(
    rho=np.array([1.0]), M=32, N=64, NFP=eq.NFP, sym=False
)
source_grid = LinearGrid(
    rho=np.array([1.0]), M=32, N=64, NFP=eq.NFP, sym=False
)
eq

In [ ]:
name = f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}.h5"
try:
    eq = load(name)
except FileNotFoundError:
    eq.solve(maxiter=300, ftol=1e-4, gtol=0, xtol=0, verbose=3);

In [ ]:
eq.save(name)

In [ ]:
plot_section(eq, "|F|_normalized", log=True);

In [ ]:
# eq.current *= -1
# eq.Psi *= -1
# eq.iota = eq.get_profile("iota")

objective = ObjectiveFunction(
    BoundaryError(
        eq=eq,
        field=field,
        field_fixed=True,
        field_grid=coil_grid,
        source_grid=source_grid,
        eval_grid=eval_grid,
        b_plasma_chunk_size=200,
        bs_chunk_size=200,
    ),
    jac_chunk_size=1,
)
constraints = (
    ForceBalance(eq=eq, grid=QuadratureGrid(L=eq.L_grid, M=eq.M_grid, N=eq.N_grid, NFP=eq.NFP)),
    FixPressure(eq=eq),
    FixPsi(eq=eq),
)
if eq.current is not None:
    constraints += (FixCurrent(eq),)
else:
    constraints += (FixIota(eq),)

In [ ]:
eq0_lowres = eq.copy()

In [ ]:
try:
    eq = load(f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-fb-k4.h5")
except FileNotFoundError:
    k = 4
    R_modes = np.vstack(
        (
            # [0, 0, 0],  # kind of cheating...
            eq.surface.R_basis.modes[np.max(np.abs(eq.surface.R_basis.modes), 1) > k, :],
        )
    )
    Z_modes = eq.surface.Z_basis.modes[np.max(np.abs(eq.surface.Z_basis.modes), 1) > k, :]
    bdry_constraints = (
        FixBoundaryR(eq=eq, modes=R_modes),
        FixBoundaryZ(eq=eq, modes=Z_modes),
    )
    eq, out = eq.optimize(
        objective,
        constraints + bdry_constraints,
        optimizer="proximal-lsq-exact",
        x_scale="ess",
        verbose=3,
        maxiter=30,
        ftol=1e-3,
        options={"solve_options":{"ftol": 1e-3, "gtol": 0, "xtol": 0, "verbose": 0}},
    )

In [ ]:
eq0_lowres_fb = eq.copy()
eq0_lowres_fb.save(f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-fb-k4.h5")

In [ ]:
plot_section(eq, "|F|_normalized", log=True);

In [ ]:
plot_comparison([eq0, eq0_lowres_fb], phi=6);

# Get Free Boundary with Low Umbilic Coil Current

In [ ]:
minor_radius = eq.compute("a")["a"]
offset = 1.4 * minor_radius
zeta = np.linspace(0, 2 * np.pi, 41)
helical_offset = 0
R0 = eq.surface.R_lmn[eq.surface.R_basis.get_idx(0, 0, 0)]
# assumes that theta = zeta - helical_offset for n/m=1 umbilic coil
# use m * theta - n * zeta = - helical_offset for general n/m
# parametrization of a curve is R = R0 + offset * cos(theta)
# and Z = offset * sin(theta)
R = R0 + offset * np.cos(zeta - helical_offset)
Z = offset * np.sin(zeta - helical_offset)

data = jnp.vstack([R, zeta, Z]).T
umbilic_coil = FourierRZCoil.from_values(
    current=-2.1e3,
    coords=data,
    N=10,
    basis="rpz",
)
coil_grid = LinearGrid(N=50)

helical_offset = np.pi
R = R0 + offset * np.cos(zeta - helical_offset)
Z = offset * np.sin(zeta - helical_offset)

data = jnp.vstack([R, zeta, Z]).T
umbilic_coil2 = FourierRZCoil.from_values(
    current=-2.1e3,
    coords=data,
    N=10,
    basis="rpz",
)

In [ ]:
plot_coil_and_surfaces(eq, coils=[umbilic_coil, umbilic_coil2] if use_coil2 else umbilic_coil)

In [ ]:
field = field0.copy()
Ic = 1000
umbilic_coil.current = Ic
umbilic_coil2.current = Ic
field.append(umbilic_coil)
if use_coil2:
    field.append(umbilic_coil2)

## Get Free Boundary for Low Umbilic Coil Current

In [ ]:
eq = eq0_lowres_fb.copy()

In [ ]:
objective = ObjectiveFunction(
    BoundaryError(
        eq=eq,
        field=field,
        field_fixed=True,
        field_grid=coil_grid,
        source_grid=source_grid,
        eval_grid=eval_grid,
        b_plasma_chunk_size=200,
        bs_chunk_size=200,
    ),
    jac_chunk_size=1,
)
constraints = (
    ForceBalance(eq=eq, grid=QuadratureGrid(L=eq.L_grid, M=eq.M_grid, N=eq.N_grid, NFP=eq.NFP)),
    FixPressure(eq=eq),
    FixPsi(eq=eq),
)
if eq.current is not None:
    constraints += (FixCurrent(eq),)
else:
    constraints += (FixIota(eq),)

In [ ]:
try:
    eq = load(f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-{2 if use_coil2 else 1}c1000-k4.h5")
except FileNotFoundError:
    k = 4
    R_modes = np.vstack(
        (
            [0, 0, 0],  # kind of cheating...
            eq.surface.R_basis.modes[np.max(np.abs(eq.surface.R_basis.modes), 1) > k, :],
        )
    )
    Z_modes = eq.surface.Z_basis.modes[np.max(np.abs(eq.surface.Z_basis.modes), 1) > k, :]
    bdry_constraints = (
        FixBoundaryR(eq=eq, modes=R_modes),
        FixBoundaryZ(eq=eq, modes=Z_modes),
    )
    eq, out = eq.optimize(
        objective,
        constraints + bdry_constraints,
        optimizer="proximal-lsq-exact",
        x_scale="ess",
        verbose=3,
        maxiter=20,
        ftol=1e-3,
        options={"solve_options": {"ftol": 1e-3, "gtol": 0, "xtol": 0, "verbose": 0}},
    )

In [ ]:
eq_coil_1000 = eq.copy()
eq_coil_1000.save(f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-{2 if use_coil2 else 1}c1000-k4.h5")

In [ ]:
plot_coil_and_surfaces(
    eq_coil_1000, coils=[umbilic_coil, umbilic_coil2] if use_coil2 else umbilic_coil
)
plot_section(eq_coil_1000, "|F|_normalized", log=True);
fig, ax = plot_1d(eq_coil_1000, "iota", color="red", figsize=(10,10))
plot_1d(eq0_lowres_fb, "iota", ax=ax, color="blue");
plot_comparison([eq0_lowres_fb, eq_coil_1000], labels=["no coil", "coil at 1000 A"], phi=6);

# Slightly higher coil current

In [ ]:
field = field0.copy()
Ic = 2000
umbilic_coil.current = Ic
umbilic_coil2.current = Ic
field.append(umbilic_coil)
if use_coil2:
    field.append(umbilic_coil2)

In [ ]:
eq = eq_coil_1000.copy()

In [ ]:
objective = ObjectiveFunction(
    BoundaryError(
        eq=eq,
        field=field,
        field_fixed=True,
        field_grid=coil_grid,
        source_grid=source_grid,
        eval_grid=eval_grid,
        b_plasma_chunk_size=200,
        bs_chunk_size=200,
    ),
    jac_chunk_size=1,
)
constraints = (
    ForceBalance(eq=eq, grid=QuadratureGrid(L=eq.L_grid, M=eq.M_grid, N=eq.N_grid, NFP=eq.NFP)),
    FixPressure(eq=eq),
    FixPsi(eq=eq),
)
if eq.current is not None:
    constraints += (FixCurrent(eq),)
else:
    constraints += (FixIota(eq),)

In [ ]:
try:
    eq = load(f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-{2 if use_coil2 else 1}c2000-k4.h5")
except FileNotFoundError:
    k = 4
    R_modes = np.vstack(
        (
            [0, 0, 0],  # kind of cheating...
            eq.surface.R_basis.modes[np.max(np.abs(eq.surface.R_basis.modes), 1) > k, :],
        )
    )
    Z_modes = eq.surface.Z_basis.modes[np.max(np.abs(eq.surface.Z_basis.modes), 1) > k, :]
    bdry_constraints = (
        FixBoundaryR(eq=eq, modes=R_modes),
        FixBoundaryZ(eq=eq, modes=Z_modes),
    )
    eq, out = eq.optimize(
        objective,
        constraints + bdry_constraints,
        optimizer="proximal-lsq-exact",
        x_scale="ess",
        verbose=3,
        maxiter=20,
        ftol=1e-3,
        options={"solve_options": {"ftol": 1e-3, "gtol": 0, "xtol": 0, "verbose": 0}},
    )

In [ ]:
eq_coil_2000 = eq.copy()
eq_coil_2000.save(f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-{2 if use_coil2 else 1}c2000-k4.h5")

In [ ]:
plot_coil_and_surfaces(
    eq_coil_2000, coils=[umbilic_coil, umbilic_coil2] if use_coil2 else umbilic_coil
)
plot_section(eq_coil_2000, "|F|_normalized", log=True);
fig, ax = plot_1d(eq_coil_2000, "iota", color="red", figsize=(10,10))
plot_1d(eq0_lowres_fb, "iota", ax=ax, color="blue");
plot_comparison([eq0_lowres_fb, eq_coil_2000], labels=["no coil", "coil at 2000 A"], phi=6);

# 3000A Current

In [ ]:
field = field0.copy()
Ic = 3000
umbilic_coil.current = Ic
umbilic_coil2.current = Ic
field.append(umbilic_coil)
if use_coil2:
    field.append(umbilic_coil2)

In [ ]:
eq = eq_coil_2000.copy()

In [ ]:
objective = ObjectiveFunction(
    BoundaryError(
        eq=eq,
        field=field,
        field_fixed=True,
        field_grid=coil_grid,
        source_grid=source_grid,
        eval_grid=eval_grid,
        b_plasma_chunk_size=200,
        bs_chunk_size=200,
    ),
    jac_chunk_size=1,
)
constraints = (
    ForceBalance(eq=eq, grid=QuadratureGrid(L=eq.L_grid, M=eq.M_grid, N=eq.N_grid, NFP=eq.NFP)),
    FixPressure(eq=eq),
    FixPsi(eq=eq),
)
if eq.current is not None:
    constraints += (FixCurrent(eq),)
else:
    constraints += (FixIota(eq),)

In [ ]:
try:
    eq = load(f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-{2 if use_coil2 else 1}c3000-k4.h5")
except FileNotFoundError:
    k = 4
    R_modes = np.vstack(
        (
            [0, 0, 0],  # kind of cheating...
            eq.surface.R_basis.modes[np.max(np.abs(eq.surface.R_basis.modes), 1) > k, :],
        )
    )
    Z_modes = eq.surface.Z_basis.modes[np.max(np.abs(eq.surface.Z_basis.modes), 1) > k, :]
    bdry_constraints = (
        FixBoundaryR(eq=eq, modes=R_modes),
        FixBoundaryZ(eq=eq, modes=Z_modes),
    )
    eq, out = eq.optimize(
        objective,
        constraints + bdry_constraints,
        optimizer="proximal-lsq-exact",
        x_scale="ess",
        verbose=3,
        maxiter=20,
        ftol=1e-3,
        options={"solve_options": {"ftol": 1e-3, "gtol": 0, "xtol": 0, "verbose": 0}},
    )

In [ ]:
eq_coil_3000 = eq.copy()
eq_coil_3000.save(f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-{2 if use_coil2 else 1}c3000-k4.h5")

In [ ]:
plot_coil_and_surfaces(
    eq_coil_3000, coils=[umbilic_coil, umbilic_coil2] if use_coil2 else umbilic_coil
)
plot_section(eq_coil_3000, "|F|_normalized", log=True);
fig, ax = plot_1d(eq_coil_3000, "iota", color="red", figsize=(10,10))
plot_1d(eq0_lowres_fb, "iota", ax=ax, color="blue");
plot_comparison([eq0_lowres_fb, eq_coil_3000], labels=["no coil", "coil at 3000 A"], phi=6);

# 4000A Current

In [ ]:
field = field0.copy()
Ic = 4000
umbilic_coil.current = Ic
umbilic_coil2.current = Ic
field.append(umbilic_coil)
if use_coil2:
    field.append(umbilic_coil2)

In [ ]:
eq = eq_coil_3000.copy()

In [ ]:
objective = ObjectiveFunction(
    BoundaryError(
        eq=eq,
        field=field,
        field_fixed=True,
        field_grid=coil_grid,
        source_grid=source_grid,
        eval_grid=eval_grid,
        b_plasma_chunk_size=200,
        bs_chunk_size=200,
    ),
    jac_chunk_size=1,
)
constraints = (
    ForceBalance(eq=eq, grid=QuadratureGrid(L=eq.L_grid, M=eq.M_grid, N=eq.N_grid, NFP=eq.NFP)),
    FixPressure(eq=eq),
    FixPsi(eq=eq),
)
if eq.current is not None:
    constraints += (FixCurrent(eq),)
else:
    constraints += (FixIota(eq),)

In [ ]:
try:
    eq = load(f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-{2 if use_coil2 else 1}c4000-k4.h5")
except FileNotFoundError:
    k = 4
    R_modes = np.vstack(
        (
            [0, 0, 0],  # kind of cheating...
            eq.surface.R_basis.modes[np.max(np.abs(eq.surface.R_basis.modes), 1) > k, :],
        )
    )
    Z_modes = eq.surface.Z_basis.modes[np.max(np.abs(eq.surface.Z_basis.modes), 1) > k, :]
    bdry_constraints = (
        FixBoundaryR(eq=eq, modes=R_modes),
        FixBoundaryZ(eq=eq, modes=Z_modes),
    )
    eq, out = eq.optimize(
        objective,
        constraints + bdry_constraints,
        optimizer="proximal-lsq-exact",
        x_scale="ess",
        verbose=3,
        maxiter=20,
        ftol=1e-3,
        options={
            "solve_options": {
                "ftol": 1e-3,
                "gtol": 0,
                "xtol": 0,
                "verbose": 0,
                "options": {"tr_method": "svd"},
            },
            "tr_method": "svd",
        },
    )

In [ ]:
eq_coil_4000 = eq.copy()
# eq_coil_4000.save(
#     f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-{2 if use_coil2 else 1}c4000-k4.h5"
# )

In [ ]:
plot_coil_and_surfaces(
    eq_coil_4000, coils=[umbilic_coil, umbilic_coil2] if use_coil2 else umbilic_coil
)
plot_section(eq_coil_4000, "|F|_normalized", log=True)
fig, ax = plot_1d(eq_coil_4000, "iota", color="red", figsize=(10, 10))
plot_1d(eq0_lowres_fb, "iota", ax=ax, color="blue");
plot_comparison(
    [eq0_lowres_fb, eq_coil_4000], labels=["no coil", "coil at 4000 A"], phi=6
);

# 5000A Current

In [ ]:
field = field0.copy()
Ic = 5000
umbilic_coil.current = Ic
umbilic_coil2.current = Ic
field.append(umbilic_coil)
if use_coil2:
    field.append(umbilic_coil2)

In [ ]:
eq = eq_coil_4000.copy()

In [ ]:
objective = ObjectiveFunction(
    BoundaryError(
        eq=eq,
        field=field,
        field_fixed=True,
        field_grid=coil_grid,
        source_grid=source_grid,
        eval_grid=eval_grid,
        b_plasma_chunk_size=200,
        bs_chunk_size=200,
    ),
    jac_chunk_size=1,
)
constraints = (
    ForceBalance(eq=eq, grid=QuadratureGrid(L=eq.L_grid, M=eq.M_grid, N=eq.N_grid, NFP=eq.NFP)),
    FixPressure(eq=eq),
    FixPsi(eq=eq),
)
if eq.current is not None:
    constraints += (FixCurrent(eq),)
else:
    constraints += (FixIota(eq),)

In [ ]:
try:
    eq = load(f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-{2 if use_coil2 else 1}c5000-k4.h5")
except FileNotFoundError:
    k = 4
    R_modes = np.vstack(
        (
            [0, 0, 0],  # kind of cheating...
            eq.surface.R_basis.modes[np.max(np.abs(eq.surface.R_basis.modes), 1) > k, :],
        )
    )
    Z_modes = eq.surface.Z_basis.modes[np.max(np.abs(eq.surface.Z_basis.modes), 1) > k, :]
    bdry_constraints = (
        FixBoundaryR(eq=eq, modes=R_modes),
        FixBoundaryZ(eq=eq, modes=Z_modes),
    )
    eq, out = eq.optimize(
        objective,
        constraints + bdry_constraints,
        optimizer="proximal-lsq-exact",
        x_scale="ess",
        verbose=3,
        maxiter=20,
        ftol=1e-3,
        options={
            "solve_options": {
                "ftol": 1e-3,
                "gtol": 0,
                "xtol": 0,
                "verbose": 0,
                "options": {"tr_method": "svd"},
            },
            "tr_method": "svd",
        },
    )

In [ ]:
eq_coil_5000 = eq.copy()
eq_coil_5000.save(
    f"./equilibria/desc-eq-HBT_105995_06-rev-curr-stage2-fb-L{eq.L}M{eq.M}N{eq.N}NFP{eq.NFP}-{2 if use_coil2 else 1}c5000-k4.h5"
)

In [ ]:
plot_coil_and_surfaces(
    eq_coil_5000, coils=[umbilic_coil, umbilic_coil2] if use_coil2 else umbilic_coil
)
plot_section(eq_coil_5000, "|F|_normalized", log=True)
fig, ax = plot_1d(eq_coil_5000, "iota", color="red", figsize=(10, 10))
plot_1d(eq0_lowres_fb, "iota", ax=ax, color="blue");
plot_comparison(
    [eq0_lowres_fb, eq_coil_5000], labels=["no coil", "coil at 5000 A"], phi=6
);

In [ ]:
fig = plot_coils(umbilic_coil)
plot_coils(umbilic_coil2, fig=fig)
plot_3d(eq_coil_5000, "curvature_k2_rho", fig=fig)

In [ ]:
eq_coil_1000